# ASISTENTE DE IA

Asistente de IA entrenado para poder tener una conversacion fluida y natural con el data set OpenAssistant/oasst1

# IMPORTAMOS EL DATASET CORRECTO

In [3]:
from datasets import load_dataset

print("Descargando OpenAssistant (oasst1)...")
# Cargamos el dataset completo
dataset = load_dataset("OpenAssistant/oasst1")

# Filtramos solo los mensajes en español
df = dataset["train"].to_pandas()
df_es = df[df["lang"] == "es"]

# Creamos pares de Pregunta (prompter) y Respuesta (assistant)
conversaciones = []
mensaje_map = df_es.set_index("message_id")["text"].to_dict()

for _, row in df_es.iterrows():
    if row["role"] == "assistant" and row["parent_id"] in mensaje_map:
        pregunta = mensaje_map[row["parent_id"]]
        respuesta = row["text"]
        texto_chat = f"<|user|>\n{pregunta}\n<|assistant|>\n{respuesta} <|endoftext|>\n"
        conversaciones.append(texto_chat)

# Guardamos el dataset conversacional
archivo_salida = "dataset_openassistant_es.txt"
with open(archivo_salida, "w", encoding="utf-8") as f:
    f.writelines(conversaciones)

print(f"¡Listo! Se guardaron {len(conversaciones)} interacciones de chat fluido en '{archivo_salida}'.")

Descargando OpenAssistant (oasst1)...


README.md:   0%|          | 0.00/10.2k [00:00<?, ?B/s]

data/train-00000-of-00001-b42a775f407cee(…): reconstructing file:   0%|          |  0.00B / 39.5MB            

data/train-00000-of-00001-b42a775f407cee(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-134b8fd0c(…): reconstructing file:   0%|          |  0.00B / 2.08MB            

data/validation-00000-of-00001-134b8fd0c(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

¡Listo! Se guardaron 15296 interacciones de chat fluido en 'dataset_openassistant_es.txt'.


# CONSTRUIMOS EL TOKENIZADOR

In [4]:
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. Configurar el Tokenizador BPE
tokenizer = Tokenizer(BPE(unk_token="<|unk|>"))
tokenizer.pre_tokenizer = Whitespace()

# Define los tokens especiales de tu interfaz de chat
special_tokens = ["<|unk|>", "<|pad|>", "<|user|>", "<|assistant|>", "<|endoftext|>"]

trainer = BpeTrainer(
    vocab_size=16000, # Vocabulario optimizado para tu Transformer de 512d
    special_tokens=special_tokens
)

# 2. Entrenar con el archivo de texto
archivos = ["dataset_openassistant_es.txt"]
tokenizer.train(archivos, trainer)

# 3. Guardar el tokenizador para reutilizarlo en la API y FastAPI
os.makedirs("tokenizer_config", exist_ok=True)
tokenizer.save("tokenizer_config/tokenizer_t800.json")

print(f"¡Tokenizador entrenado con éxito! Tamaño del vocabulario: {tokenizer.get_vocab_size()}")

¡Tokenizador entrenado con éxito! Tamaño del vocabulario: 16000


Los embeddings

In [5]:
import torch
import torch.nn as nn

class TransformerEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model=512, max_seq_len=64, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # Embedding de palabras
        self.token_embeddings = nn.Embedding(vocab_size, d_model)

        # Embedding posicional (aprendible)
        self.position_embeddings = nn.Embedding(max_seq_len, d_model)

        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, input_ids):
        seq_len = input_ids.size(1)

        # Generar posiciones [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(0, seq_len, dtype=torch.long, device=input_ids.device).unsqueeze(0)

        # Sumar Token Embedding + Position Embedding
        tok_emb = self.token_embeddings(input_ids)
        pos_emb = self.position_embeddings(positions)

        x = tok_emb + pos_emb
        x = self.layer_norm(x)
        return self.dropout(x)

# --- Prueba rápida de funcionamiento ---
vocab_size = tokenizer.get_vocab_size()
embeddings_layer = TransformerEmbeddings(vocab_size=vocab_size, d_model=512, max_seq_len=64)

# Simulamos un lote de 2 frases con longitud 64
ejemplo_ids = torch.randint(0, vocab_size, (2, 64))
salida = embeddings_layer(ejemplo_ids)

print(f"Forma de la salida de Embeddings: {salida.shape}")
# Debería imprimir: torch.Size([2, 64, 512])

Forma de la salida de Embeddings: torch.Size([2, 64, 512])


# MULTIHEAD ATTENTION

In [6]:
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model debe ser divisible entre n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        # Proyectar y separar en n_heads
        q = self.q_linear(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_linear(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_linear(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        # Atención Escalada por Producto Punto
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = torch.matmul(attn_weights, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out_proj(out)

class FeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, dropout=dropout)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x

# Empezemos el entrenamiento.

In [8]:
import torch
import torch.nn as nn

class XelulaLM(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, num_layers=6, max_seq_len=64, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.embeddings = TransformerEmbeddings(vocab_size, d_model, max_seq_len, dropout)

        # Bloques apilados del Transformer Decoder
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout) for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def generate_causal_mask(self, seq_len, device):
        # Máscara triangular superior para evitar que el modelo "vea el futuro"
        mask = torch.tril(torch.ones((seq_len, seq_len), device=device)).bool()
        return mask.unsqueeze(0).unsqueeze(1) # [1, 1, seq_len, seq_len]

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        device = input_ids.device

        mask = self.generate_causal_mask(seq_len, device)
        x = self.embeddings(input_ids)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

# ANTES DE ENTRENAR CONECTAMOS A GOOGLE DRIVE PARA TENER CHECK POINTS DEL ENTRENAMIENTO

In [9]:
from google.colab import drive
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# 1. Montar Google Drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/XELULA_MODEL"
os.makedirs(drive_path, exist_ok=True)

# ==========================================
# REEMPLAZA ESTA CLASE EN TU COLAB
# ==========================================
class ChatDataset(Dataset):
    def __init__(self, text_file, tokenizer, max_seq_len=64):
        with open(text_file, "r", encoding="utf-8") as f:
            text = f.read()
        encoded = tokenizer.encode(text)
        self.tokens = torch.tensor(encoded.ids, dtype=torch.long)
        self.max_seq_len = max_seq_len

    def __len__(self):
        # Salta de 64 en 64 tokens para optimizar el tamaño de la época
        return (len(self.tokens) - 1) // self.max_seq_len

    def __getitem__(self, idx):
        start_idx = idx * self.max_seq_len
        chunk = self.tokens[start_idx : start_idx + self.max_seq_len + 1]

        if len(chunk) < self.max_seq_len + 1:
            chunk = self.tokens[-self.max_seq_len - 1:]

        return chunk[:-1], chunk[1:]

# ==========================================
# LUEGO REINSTANCIAS EL DATALOADER Y CORRES EL ENTRENAMIENTO
# ==========================================
dataset_chat = ChatDataset("dataset_openassistant_es.txt", tokenizer, max_seq_len=64)
dataloader = DataLoader(dataset_chat, batch_size=32, shuffle=True)

# 4. Inicializar Modelo, Optimizador y GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = XelulaLM(vocab_size=tokenizer.get_vocab_size(), d_model=512, n_heads=8, num_layers=6, max_seq_len=64).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.token_to_id("<|pad|>"))
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

print(f"Modelo Xelula cargado en: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Modelo Xelula cargado en: cuda


# EMPEZEMOS AHORA SI CON EL ENTRENAMIENTO CON UN TOTAL DE 14 EPOCAS , SIN EMBARGO CALCULANDO SIEMPRE UN ERROR OBJETIVO DE 1.8

In [11]:
from tqdm.auto import tqdm
import os
import torch

# Parámetros del entrenamiento
epochs = 17
target_loss = 1.8

print(f"🚀 Iniciando entrenamiento de Xelula ({epochs} épocas)...")
print(f"🎯 Meta: Alcanzar Loss <= {target_loss}\n")

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    # Barra de progreso para los lotes de la época actual
    progress_bar = tqdm(dataloader, desc=f"Época {epoch+1}/{epochs}", leave=True)

    for x, y in progress_bar:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)

        # CrossEntropyLoss calculada sobre las etiquetas shifteadas
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()

        # Estabilización de gradientes
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        # Actualización de la barra en tiempo real
        progress_bar.set_postfix({"Loss actual": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(dataloader)
    print(f"✨ Fin Época {epoch+1} | Loss Promedio: {avg_loss:.4f}\n")

    # Guardar Checkpoint en Google Drive
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss
    }
    torch.save(checkpoint, os.path.join(drive_path, "xelula_checkpoint.pth"))

    # Parada temprana al alcanzar el objetivo de 1.8
    if avg_loss <= target_loss:
        print(f"🎉 ¡Objetivo alcanzado! Loss promedio ({avg_loss:.4f}) <= {target_loss}.")
        torch.save(model.state_dict(), os.path.join(drive_path, "xelula_final.pth"))
        print("💾 Modelo guardado como 'xelula_final.pth' en Google Drive.")
        break

# Guardado al completar las 14 épocas si no alcanzó 1.8 antes
if avg_loss > target_loss:
    torch.save(model.state_dict(), os.path.join(drive_path, "xelula_final.pth"))
    print("💾 14 Épocas completadas. Modelo guardado como 'xelula_final.pth' en Google Drive.")

🚀 Iniciando entrenamiento de Xelula (17 épocas)...
🎯 Meta: Alcanzar Loss <= 1.8



Época 1/17:   0%|          | 0/1490 [00:00<?, ?it/s]

KeyboardInterrupt: 

# AHORA CARGUEMOS Y PROBEMOS EL MODELO CREANDO UNA INTERFAZ GRAFICA PARA EL MISMO.

In [ ]:
!pip install -q gradio

import torch
import torch.nn.functional as F
import gradio as gr
import os

# ==========================================
# 1. CARGAR MODELO Y TOKENIZADOR
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instanciar arquitectura
model = XelulaLM(vocab_size=vocab_size, d_model=512, n_heads=8, num_layers=6, max_seq_len=64).to(device)

# Cargar pesos guardados en Google Drive (xelula_final.pth)
model_path = os.path.join(drive_path, "xelula_final.pth")
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("🤖 Pesos de Xelula cargados con éxito desde Google Drive.")
else:
    print("⚠️ No se encontró 'xelula_final.pth', usando el modelo actual en memoria.")

model.eval()

# ==========================================
# 2. MOTOR DE GENERACIÓN DE TEXTO (SAMPLING)
# ==========================================
def responder_t800(prompt_usuario, temperatura=0.5, top_p=0.8, repetition_penalty=1.2):
    if not prompt_usuario.strip():
        return "CYBERDYNE SYSTEM ERROR: ENTRADA VACÍA."

    # Formatear el prompt con las etiquetas de instrucción
   # Forzar al modelo a tomar el rol de asistente directo
    prompt_format = f"<|user|>\n{prompt_usuario.strip()}\n<|assistant|>\nHola,"
    input_ids = torch.tensor([tokenizer.encode(prompt_format).ids], device=device)

    for _ in range(50):
        inputs = input_ids[:, -64:]

        with torch.no_grad():
            logits = model(inputs)
            next_token_logits = logits[0, -1, :]

            # --- APLICAR REPETITION PENALTY ---
            for token_id in set(input_ids[0].tolist()):
                if next_token_logits[token_id] < 0:
                    next_token_logits[token_id] *= repetition_penalty
                else:
                    next_token_logits[token_id] /= repetition_penalty

            # Ajustar por temperatura
            next_token_logits = next_token_logits / temperatura

            # --- FILTRADO TOP-P (NUCLEUS SAMPLING) ---
            sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            next_token_logits[indices_to_remove] = float('-inf')

            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

            # Frenar al emitir token de fin
            if next_token.item() == tokenizer.token_to_id("<|endoftext|>"):
                break

   # Decodificar todo el vector completo
    texto_completo = tokenizer.decode(input_ids[0].tolist())

    # Extraer estrictamente lo que esté después de <|assistant|>
    if "<|assistant|>" in texto_completo:
        respuesta = texto_completo.split("<|assistant|>")[-1]
        # Limpiar tokens de control y saltos de línea sobrantes
        respuesta_limpia = respuesta.replace("<|endoftext|>", "").strip()
        return respuesta_limpia if respuesta_limpia else "[SISTEMA SIN RESPUESTA]"

    return texto_completo

# ==========================================
# 3. INTERFAZ GRÁFICA ESTILO TERMINAL T-800
# ==========================================
css_t800 = """
body, .gradio-container {
    background-color: #050505 !important;
    color: #00ff66 !important;
    font-family: 'Courier New', monospace !important;
}
textarea, input {
    background-color: #111111 !important;
    color: #00ff66 !important;
    border: 1px solid #00ff66 !important;
    font-family: 'Courier New', monospace !important;
}
button {
    background-color: #8b0000 !important;
    color: #ffffff !important;
    border: 1px solid #ff0000 !important;
    font-weight: bold !important;
}
button:hover {
    background-color: #ff0000 !important;
    box-shadow: 0 0 10px #ff0000;
}
"""

with gr.Blocks(css=css_t800, title="CYBERDYNE SYSTEMS - T-800") as demo:
    gr.Markdown(
        """
        # 🤖 CYBERDYNE SYSTEMS MODEL T-800 (XELULA CORE)
        ### SYSTEM STATUS: ONLINE | LOSS: 1.76 | TARGET ACQUIRED
        ---
        """
    )

    with gr.Row():
        with gr.Column(scale=2):
            entrada = gr.Textbox(
                label="[ INPUT DATA / INSTRUCCIÓN ]",
                placeholder="Escribe tu mensaje al T-800 aquí...",
                lines=2
            )
            btn_enviar = gr.Button("EJECUTAR RESPUESTA 🔴")

        with gr.Column(scale=1):
            temp_slider = gr.Slider(0.2, 1.2, value=0.7, step=0.1, label="Temperatura (Creatividad)")
            topp_slider = gr.Slider(0.5, 0.95, value=0.85, step=0.05, label="Top-P (Fluidez)")

    salida = gr.Textbox(
        label="[ T-800 RESPONSE / TELEMETRÍA ]",
        lines=4,
        interactive=False
    )

    btn_enviar.click(
        fn=responder_t800,
        inputs=[entrada, temp_slider, topp_slider],
        outputs=salida
    )

demo.launch(share=True, debug=True)

🤖 Pesos de Xelula cargados con éxito desde Google Drive.


/tmp/ipykernel_1499/1251644328.py:114: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css_t800, title="CYBERDYNE SYSTEMS - T-800") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://323e7e9300425ae344.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
